In [ ]:
import sys; sys.path.append('..')
import MeshFEM
import mesh, elastic_sheet, energy, benchmark
import triangulation
from tri_mesh_viewer import TriMeshViewer
import numpy as np
import mesh_operations

In [ ]:
# Generate a mesh of the unit disk
radius = 10 # 10mm
thetas = np.linspace(0, 2 * np.pi, 100, endpoint=False)
m = mesh.Mesh(*triangulation.triangulate(
    radius * np.array([np.cos(thetas), np.sin(thetas)]).T,
    np.array([np.arange(len(thetas)), np.roll(np.arange(len(thetas)), -1)]).T, outputPointMarkers=False, triArea=0.025))

# Glue mesh to itself along the boundary by creating a merged mesh
# where all internal vertices have been perturbed imperceptibly
# out-of-plane to prevent their merging.
isInternal = np.ones(m.numVertices(), dtype=bool)
isInternal[m.boundaryVertices()] = False

Vtop, Vbot = m.vertices().copy(), m.vertices().copy()
Vbot = m.vertices().copy()
Vtop[isInternal, 2] =  1e-16 # np.sqrt(1 - np.linalg.norm(Vtop[isInternal, :] / radius, axis=1)**2)
Vbot[isInternal, 2] = -1e-16 # -np.sqrt(1 - np.linalg.norm(Vtop[isInternal, :] / radius, axis=1)**2)

Ftop, Fbot = m.elements(), m.elements().copy()
Fbot[:, [0, 1]] = Fbot[:, [1, 0]]

m = mesh.Mesh(*mesh_operations.mergedMesh([(Vtop, Ftop), (Vbot, Fbot)]))

In [ ]:
#psi = energy.OptionalTensionFieldEnergy(200) # Young's modulus = 200MPa -- doesn't work well, likely because of the high bulk modulus
softPsi  = energy.NeoHookeanYoungPoisson(2, E=200, nu=0.3)  # Young's modulus = 200MPa,  Poisson's ratio = 0.3
stiffPsi = energy.NeoHookeanYoungPoisson(2, E=20000, nu=0.3) # Young's modulus = 20GPa, Poisson's ratio = 0.3
es = elastic_sheet.ElasticSheet(m, softPsi)
es.thickness = 0.05 # 0.05mm

In [ ]:
HETEROGENEOUS = True
if HETEROGENEOUS:
    es.setMaterials([softPsi, stiffPsi])
    # Assign stiffMaterial to the right half
    es.elementMaterialAssignments = m.vertices()[m.elements(), 0].mean(axis=1) < 0

In [ ]:
# Eliminate the creased seam by making each element flat in the rest state
# (optionally skip this cell to simulate the crease)
es.programFlatRestCurvature()

In [ ]:
esview = TriMeshViewer(es, wireframe=True, width=1024, height=768, scalarField=es.elementMaterialAssignments if HETEROGENEOUS else None)
if not HETEROGENEOUS:
    esview.materialLibrary.material(False).color='#33EE00'
esview.show()

In [ ]:
import py_newton_optimizer
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.niter = 500
opts.hessianProjectionController = py_newton_optimizer.HessianProjectionAdaptive()
es.hessianProjectionType = es.hessianProjectionType.MembraneFBased

In [ ]:
# Configure visualization
visNormals = False
nview = None
def updateNormalView():
    from tri_mesh_viewer import PointCloudViewer
    global nview
    if not visNormals: return
    esview.subViews = []
    nview = PointCloudViewer(es.edgeMidpoints(), vectorField=es.midedgeNormals(), superView=esview)
    nview.arrowSize = 30
updateNormalView()

# Callback to update visualization during equilibrium solve
def iter_cb(prob, it):
    if (it % 5 == 1):
       esview.update(scalarField=es.elementMaterialAssignments if HETEROGENEOUS else None)
       updateNormalView()

In [ ]:
import loads
inflation = loads.Inflation(es)

In [ ]:
inflation.pressure = 0.05
benchmark.reset()
pinVars, pinVerts = es.prepareRigidMotionPins()
es.computeEquilibrium(loads=[inflation], fixedVars=pinVars, cb=iter_cb, opts=opts)
benchmark.report()

## Finite Difference Validation

In [ ]:
import fd_validation
prob = es.EquilibriumProblem([inflation])

In [ ]:
fd_validation.gradConvergencePlot(prob)

In [ ]:
prob.invalidateCachedHessian()
fd_validation.hessConvergencePlot(prob)